In [3]:
import pandas as pd
import re
from google import genai
from dotenv import load_dotenv
import json
import re
import ast

load_dotenv('../../backend/.env')
client = genai.Client()
def get_payments_df(transactions_df:pd.DataFrame) -> pd.DataFrame:
    """separates actual payments from just transactions"""
    def find_transaction_object(text):
        pattern = r"ობიექტი:\s*([^,]+)"
        res = re.search(pattern, text)
        if res:
            return res.group(1)

    payments_df = transactions_df.loc[transactions_df["transaction_type"] == "გადახდა"] # separate actal payments from transactions
    payments_df["transaction_object"] = payments_df["დანიშნულება"].apply(lambda x:find_transaction_object(x) ) # separate where transaction was actually performed
    payments_df["transaction_object"] = payments_df["transaction_object"].fillna(value="gadakhda") # fill wherever object was unavailable
    payments_df.drop(columns=["transaction_type"],inplace=True)
    payments_df.fillna(value=0, inplace=True) # fill other NaN values
    payments_df["transaction_object"] = payments_df["transaction_object"].apply(lambda x: x.lower()) # convert to lowercase

    return payments_df

categorization_sys_prompt = '''
You are a financial transaction categorization engine. You will receive a Python list of merchant names from bank transactions and classify each one into the correct spending category.

## CATEGORIES
Classify each merchant into exactly one of the following categories:
- `Groceries` - supermarkets, food stores, fresh markets
- `Dining` - restaurants, cafes, fast food, takeaway, food delivery apps
- `Fuel` - petrol stations, gas stations, merchants ending with "gazi"
- `Transport` - public transport, taxis, ride-hailing (Uber, Bolt), parking
- `Travel` - flights, hotels, booking platforms, car rentals, travel agencies
- `Pharmacy` - pharmacies, drugstores, medical supplies
- `Healthcare` - clinics, hospitals, doctors, dentists, labs
- `Entertainment` - cinemas, concerts, events, gaming, streaming services
- `Shopping` - clothing, electronics, home goods, online marketplaces
- `Subscriptions` - recurring software, SaaS, memberships, apps
- `Utilities` - electricity, water, gas, internet, phone bills
- `Education` - courses, books, tuition, e-learning platforms
- `Fitness` - gyms, sports equipment, wellness apps
- `PersonalCare` - salons, barbers, beauty products, spas
- `Finance` - bank fees, insurance, investments, loan repayments
- `Other` - anything that does not clearly fit the above

## RULES
1. Use your real-world knowledge of brands, chains, and services to classify.
2. If a name is ambiguous, pick the most statistically likely category for that merchant type.
3. Never leave a category blank — always assign the best possible match.
4. If genuinely uncertain, use `Other`.
5. Do NOT invent new categories beyond the list above.
6. DO NOT CHANGE MERCHANT NAMES. when putting the name in the category, use the original name.

## OUTPUT FORMAT
Return ONLY a valid Python dictionary — no explanation, no markdown, no code fences.
Exact structure:

{
  "results": [
    {"merchant": "<original name>", "category": "<category>", "confidence": "<high|medium|low>"},
    ...
  ]
}

## INPUT
{merchants_list}'''


def run_llm(input_prompt, config ):
    
    response = client.models.generate_content(model="gemini-3-flash-preview",
                                              contents=input_prompt,
                                              config=config)
    return response


def clean_response(text, parse_as="json"):
    """
    Cleans LLM output by removing code fences and parses it.
    
    Args:
        text (str): The LLM response text.
        parse_as (str): "json" or "python". Determines parser.
        
    Returns:
        Parsed Python object (dict/list/etc.).
    """
    # Remove ```json, ```python, or ``` code fences
    cleaned = re.sub(r'```(?:json|python)?', '', text).strip()
    
    if parse_as == "json":
        return json.loads(cleaned)
    elif parse_as == "python":
        return ast.literal_eval(cleaned)
    else:
        raise ValueError("parse_as must be 'json' or 'python'")


def parse_categorization_response(response):
    try:
        response_text = response.text
        response_text = clean_response(response_text, parse_as="json")
        print(response_text)
        categories_df = pd.DataFrame(response_text["results"])
    except Exception as e:
        print(f"Error parsing categorization response: {e}")
        return pd.DataFrame()

    return categories_df






In [4]:

transactions_df = pd.read_excel("../../data/raw/amonaweri.xlsx", sheet_name="ტრანზაქციები")
transactions_df.drop(columns=["Unnamed: 2"],inplace=True) # drop unneccessary col
transactions_df["transaction_type"] = transactions_df["დანიშნულება"].apply(lambda x: x.split()[0]) # separate by transaction type

df = get_payments_df(transactions_df)

merchants = list(df.transaction_object.unique())
formatted = "\n".join(f"{i+1}. {m}" for i, m in enumerate(merchants))

categorization_sys_prompt = categorization_sys_prompt.replace("{merchants_list}", formatted)
print(merchants)

config = genai.types.GenerateContentConfig(
            system_instruction=categorization_sys_prompt
        )

response = run_llm(merchants, config)
merchants_df = parse_categorization_response(response)

c:\Users\Saba\.conda\envs\financeAI\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


['libre', 'gadakhda', 'nikora', 'shps gama 2023', 'spar', 'magniti', 'ori nabiji', 'shemogevle', 'ie goderdzi apkhazashvili', 'tea house veris bagi', 'pablisit service', 'agrohub', 'carrefour', "mcdonald's", 'pipes', 'bunebrivi gazi', 'tsiskvili market', 'neogas', 'gama', 'wolt', 'paiberi', 'oneprice', 'i/m zaira chichinadze', 'billiard hall', 'georgian post', 'vip pay', 'yandex.go', 'ji ji house 3', 'ltd build more', 'pegasus pegasus', 'piatto', 'evex', 'zgapari', 'ltd mp development', 'omega gazi', 'pharmadepot', 'petra beach', 'wissol', 'socar', 'daily 157', 'daily', 'psp', 'aversi 95', 'market spar', 'xs toys']
{'results': [{'merchant': 'libre', 'category': 'Groceries', 'confidence': 'high'}, {'merchant': 'gadakhda', 'category': 'Utilities', 'confidence': 'medium'}, {'merchant': 'nikora', 'category': 'Groceries', 'confidence': 'high'}, {'merchant': 'shps gama 2023', 'category': 'Healthcare', 'confidence': 'high'}, {'merchant': 'spar', 'category': 'Groceries', 'confidence': 'high'},

In [5]:
merchants_df.to_csv("../../data/processed/merchants_df.csv", index=False)


In [6]:
merchants.append("carservice")

In [7]:
def clean_input(merchants_df:pd.DataFrame, merchants_list:list):
    existing_merchants = list(merchants_df["merchant"].unique())
    merchants_list = [merchant for merchant in merchants_list if merchant not in existing_merchants]
    return merchants_list 

In [8]:
clean_input(merchants_df, merchants)

['carservice']